# CV Search Advanced Grid와 Random의 전체 24개 후보 평가 행을 합치고, 평균 허용폭과 탐색 예산 변화가 후보 선택에 미치는 영향을 분석했습니다. 이 노트북에서는 기본 실습에서 사용한 holdout을 다시 평가하지 않습니다. 심화 실습 완료. 강의 문제 원문은 제외

In [1]:
# 공통 준비 코드

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

RANDOM_STATE = 42
MEAN_TOLERANCE = 0.01
SELECTION_ORDER = [
    "std_AP",
    "n_estimators",
    "max_depth",
    "search",
]

# [1] 기본 실습과 같은 불균형 합성 데이터를 만듭니다.
X, y = make_classification(
    n_samples=1600,
    n_features=14,
    n_informative=8,
    n_redundant=2,
    weights=[0.88, 0.12],
    class_sep=0.9,
    random_state=RANDOM_STATE,
)

# [2] 기본 실습과 동일한 개발 데이터를 재현하기 위해 같은 8:2 분할을 수행합니다.
# holdout은 이 심화 페이지에서 점수 계산이나 후보 비교에 사용하지 않습니다.
X_dev, X_holdout_unused, y_dev, y_holdout_unused = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# [3] 모든 후보가 동일한 네 fold를 사용합니다.
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=RANDOM_STATE,
)

base_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

grid = GridSearchCV(
    base_model,
    {
        "n_estimators": [80, 140, 200],
        "max_depth": [4, 6, 8, 10],
    },
    scoring="average_precision",
    cv=cv,
    n_jobs=1,
    refit=False,
)

random = RandomizedSearchCV(
    base_model,
    {
        "n_estimators": list(range(80, 201, 10)),
        "max_depth": list(range(4, 11)),
    },
    n_iter=12,
    scoring="average_precision",
    cv=cv,
    n_jobs=1,
    random_state=RANDOM_STATE,
    refit=False,
)

searches = {"Grid": grid, "Random": random}
for search in searches.values():
    search.fit(X_dev, y_dev)

print("Grid/Random candidates:", [
    len(search.cv_results_["params"])
    for search in searches.values()
])
print("total CV fits:", sum(
    len(search.cv_results_["params"]) * cv.get_n_splits()
    for search in searches.values()
))

assert [len(search.cv_results_["params"]) for search in searches.values()] == [12, 12]

Grid/Random candidates: [12, 12]
total CV fits: 96


In [2]:
def collect_all_candidates(fitted_searches: dict) -> pd.DataFrame:
    """Grid와 Random의 모든 CV 후보 행을 하나의 표로 합칩니다."""
    candidate_rows = []

    for search_name, fitted_search in fitted_searches.items():
        # cv_results_를 DataFrame으로 바꾸면 후보 하나가 한 행이 됩니다.
        results = pd.DataFrame(fitted_search.cv_results_)

        for _, row in results.iterrows():
            candidate_rows.append({
                "search": search_name,
                "mean_AP": float(row["mean_test_score"]),
                "std_AP": float(row["std_test_score"]),
                "n_estimators": int(row["param_n_estimators"]),
                "max_depth": int(row["param_max_depth"]),
            })

    candidates = pd.DataFrame(candidate_rows)
    assert len(candidates) == 24
    return candidates


def select_candidate(
    candidates: pd.DataFrame,
    mean_tolerance: float = MEAN_TOLERANCE,
):
    """test 없이 사전 규칙으로 후보 하나를 고정합니다."""
    # 전체 24개 후보에서 가장 높은 평균 AP를 찾습니다.
    best_mean = float(candidates["mean_AP"].max())

    # 최고 평균과 충분히 가까운 후보만 다음 단계로 보냅니다.
    eligible = candidates[
        candidates["mean_AP"] >= best_mean - mean_tolerance
    ].copy()

    # 평균 허용 범위 안에서는 변동 → 트리 수 → 깊이 → 이름 순서로 선택합니다.
    selected = eligible.sort_values(
        SELECTION_ORDER,
        kind="mergesort",
    ).iloc[0]

    return selected, eligible, best_mean


candidates = collect_all_candidates(searches)
selected, eligible, best_mean = select_candidate(candidates)

# 24개는 탐색법별 평가 결과 행 수입니다.
# 같은 하이퍼파라미터 좌표가 Grid와 Random에서 중복될 수 있으므로
# 고유 파라미터 조합 수도 별도로 확인합니다.
unique_parameter_count = len(
    candidates[["n_estimators", "max_depth"]].drop_duplicates()
)

print("evaluation rows:", len(candidates))
print("unique parameter combinations:", unique_parameter_count)
print("best mean AP:", f"{best_mean:.3f}")
print("eligible candidates:", len(eligible))
print("selected:", {
    "search": selected["search"],
    "n_estimators": int(selected["n_estimators"]),
    "max_depth": int(selected["max_depth"]),
})
print(
    "selected CV mean/std:",
    f"{selected['mean_AP']:.3f}",
    f"{selected['std_AP']:.3f}",
)
print("sealed test reused:", False)

assert len(candidates) == 24
assert unique_parameter_count <= len(candidates)
assert float(selected["mean_AP"]) >= best_mean - MEAN_TOLERANCE

evaluation rows: 24
unique parameter combinations: 22
best mean AP: 0.618
eligible candidates: 5
selected: {'search': 'Grid', 'n_estimators': 80, 'max_depth': 10}
selected CV mean/std: 0.613 0.030
sealed test reused: False


In [3]:
def analyze_tolerance_sensitivity(candidates, tolerances):
    rows = []

    for tolerance in tolerances:
        selected, eligible, _ = select_candidate(
            candidates,
            mean_tolerance=tolerance,
        )
        rows.append({
            "tolerance": tolerance,
            "eligible": len(eligible),
            "search": selected["search"],
            "mean_AP": float(selected["mean_AP"]),
            "std_AP": float(selected["std_AP"]),
            "n_estimators": int(selected["n_estimators"]),
            "max_depth": int(selected["max_depth"]),
        })

    return pd.DataFrame(rows)


tolerance_table = analyze_tolerance_sensitivity(
    candidates,
    [0.000, 0.005, 0.010, 0.020],
)

print(tolerance_table.to_string(
    index=False,
    formatters={
        "mean_AP": "{:.3f}".format,
        "std_AP": "{:.3f}".format,
    },
))

assert tolerance_table["eligible"].is_monotonic_increasing

 tolerance  eligible search mean_AP std_AP  n_estimators  max_depth
     0.000         1   Grid   0.618  0.042           200         10
     0.005         4 Random   0.613  0.038           150         10
     0.010         5   Grid   0.613  0.030            80         10
     0.020         5   Grid   0.613  0.030            80         10


In [4]:
def summarize_random_budget(name, search, cv):
    best_index = search.best_index_
    candidate_count = len(search.cv_results_["params"])
    return {
        "budget": name,
        "candidates": candidate_count,
        "CV_fits": candidate_count * cv.get_n_splits(),
        "best_mean_AP": float(
            search.cv_results_["mean_test_score"][best_index]
        ),
        "best_std_AP": float(
            search.cv_results_["std_test_score"][best_index]
        ),
        "best_params": search.cv_results_["params"][best_index],
    }


# 같은 공간과 평가 조건에서 후보 수만 6개로 줄입니다.
random_6 = RandomizedSearchCV(
    base_model,
    {
        "n_estimators": list(range(80, 201, 10)),
        "max_depth": list(range(4, 11)),
    },
    n_iter=6,
    scoring="average_precision",
    cv=cv,
    n_jobs=1,
    random_state=RANDOM_STATE,
    refit=False,
)
random_6.fit(X_dev, y_dev)

budget_comparison = pd.DataFrame([
    summarize_random_budget("Random-6", random_6, cv),
    summarize_random_budget("Random-12", random, cv),
])

print(budget_comparison.to_string(
    index=False,
    formatters={
        "best_mean_AP": "{:.3f}".format,
        "best_std_AP": "{:.3f}".format,
    },
))

assert budget_comparison["candidates"].tolist() == [6, 12]
assert budget_comparison["CV_fits"].tolist() == [24, 48]

   budget  candidates  CV_fits best_mean_AP best_std_AP                            best_params
 Random-6           6       24        0.616       0.042 {'n_estimators': 180, 'max_depth': 10}
Random-12          12       48        0.616       0.042 {'n_estimators': 180, 'max_depth': 10}
